# Validation hybrid decoding final gate before v2

Cache-only evaluation of four targeted hybrid decoding candidates. This notebook performs no training, model loading, checkpoint loading, or model forward pass. It reads only the frozen validation dataset, codon reference, and the completed finetuned validation decoding cache. The test split is prohibited. A GPU is not required.

## Clone the private repository securely
Store a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`. Authentication uses `GIT_ASKPASS`; the token is not printed or embedded in a command.

In [ ]:
import hashlib
import json
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/Yuano0o/codontransformer-nb.git"
PROJECT_DIR = Path("/content/codontransformer-project")
UPSTREAM_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass = Path(askpass_dir) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
        '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\nesac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
    if not (PROJECT_DIR / ".git").is_dir():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=env)
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=env)
    env.pop("COLAB_GITHUB_TOKEN", None)
github_token = None
os.chdir(PROJECT_DIR)
UPSTREAM_DIR = PROJECT_DIR / "upstream"
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", UPSTREAM_COMMIT], check=True)

## Install analysis dependencies and mount Google Drive

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (str(UPSTREAM_DIR), os.environ.get("PYTHONPATH", ""))))
if str(UPSTREAM_DIR) not in os.sys.path:
    os.sys.path.insert(0, str(UPSTREAM_DIR))
from google.colab import drive
drive.mount("/content/drive")

## Verify frozen validation inputs and the completed source cache

The source manifest must match the 531-record validation hash and the exact v1 checkpoint SHA256. Only the finetuned record cache is read.

In [ ]:
import yaml
CONFIG_PATH = PROJECT_DIR / "configs/check_validation_hybrid_decoding.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert CONFIG["dataset_role"] == "validation"
DRIVE_ROOT = Path("/content/drive/MyDrive/CodonTransformer")
VALIDATION_PATH = DRIVE_ROOT / CONFIG["paths"]["validation_dataset"]
REFERENCE_PATH = DRIVE_ROOT / CONFIG["paths"]["codon_reference"]
RUN_DIR = DRIVE_ROOT / CONFIG["paths"]["formal_run"]
SOURCE_DIR = RUN_DIR / CONFIG["paths"]["source_decoding_directory"]
OUTPUT_DIR = RUN_DIR / CONFIG["paths"]["output_directory"]
SOURCE_MANIFEST = SOURCE_DIR / "evaluation_manifest.json"
FINETUNED_CACHE = SOURCE_DIR / "record_cache/finetuned.jsonl"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for label, path in (("validation", VALIDATION_PATH), ("reference", REFERENCE_PATH), ("source manifest", SOURCE_MANIFEST), ("finetuned cache", FINETUNED_CACHE)):
    assert path.is_file(), f"Missing {label}: {path}"
with VALIDATION_PATH.open(encoding="utf-8") as handle:
    assert sum(bool(line.strip()) for line in handle) == 531
assert sha256(VALIDATION_PATH) == CONFIG["inputs"]["validation_sha256"]
assert sha256(REFERENCE_PATH) == CONFIG["inputs"]["reference_sha256"]
source_manifest = json.loads(SOURCE_MANIFEST.read_text(encoding="utf-8"))
assert source_manifest["dataset_role"] == "validation"
assert source_manifest["checkpoint_sha256"] == CONFIG["inputs"]["source_checkpoint_sha256"]
print("Source cache records:", sum(bool(line.strip()) for line in FINETUNED_CACHE.open(encoding="utf-8")))
print("Persistent output:", OUTPUT_DIR)

## Run the cache-only decoder gate

The command splices cached synonymous codons, verifies every translation, applies paired bootstrap/Wilcoxon/BH statistics overall and by frozen length strata, and decides whether any decoder-only candidate passes.

In [ ]:
subprocess.run([
    os.sys.executable, "scripts/check_validation_hybrid_decoding.py",
    "--config", str(CONFIG_PATH),
    "--validation-dataset", str(VALIDATION_PATH),
    "--reference-json", str(REFERENCE_PATH),
    "--source-decoding-dir", str(SOURCE_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--force",
], check=True)

## Inspect the final decoder-only decision

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / "hybrid_decoding_summary.json"
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
assert summary["dataset_role"] == "validation"
assert summary["records"] == 531
assert summary["test_access_prohibited"] is True
assert summary["model_forward_performed"] is False
assert summary["training_performed"] is False
assert "proceed_to_v2_finetuning" in summary["decision"]
print(json.dumps(summary["decision"], indent=2))
print((OUTPUT_DIR / "hybrid_decoding_report.md").read_text())
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)